In [ ]:
import os
import pandas as pd
import lyricsgenius
import time
import random
from dotenv import load_dotenv


# -----------------------------
# CONFIGURATION
# -----------------------------

# Load environment variables from .env file (using absolute path for reliability)
env_path = os.path.join(os.path.dirname(os.path.dirname(os.path.abspath('__file__'))), '.env')
load_dotenv(env_path, override=True, encoding='utf-8')

# 1) Genius API Token
GENIUS_API_TOKEN = os.getenv("GENIUS_API_TOKEN")
if not GENIUS_API_TOKEN:
    raise ValueError("GENIUS_API_TOKEN not found in .env file")

# 2) Path to your artist list (one artist name per line)
ARTIST_LIST_PATH = "artists.txt"

# 3) Output CSV (where we'll append results as we go)
OUTPUT_CSV = "scraped_lyrics.csv"


# 4) How many songs to fetch per artist
SONGS_PER_ARTIST = int(os.getenv("SONGS_PER_ARTIST", "25"))
print(f"Will fetch up to {SONGS_PER_ARTIST} songs per artist")

# 5) Pause (seconds) between artist requests to avoid rate-limiting
SLEEP_BETWEEN_ARTISTS = float(os.getenv("SLEEP_BETWEEN_ARTISTS", "1.5"))
print(f"Will sleep {SLEEP_BETWEEN_ARTISTS} seconds between artist requests")

# 6) Rate limit handling configuration
INITIAL_BACKOFF = int(os.getenv("INITIAL_BACKOFF", 10))  # Start with 10 seconds
MAX_RETRIES = int(os.getenv("MAX_RETRIES", 5))       # Try up to 5 times

print(f"Using initial backoff of {INITIAL_BACKOFF}s with {MAX_RETRIES} max retries")
# -----------------------------
# INITIALIZE GENIUS CLIENT
# -----------------------------

# Initialize lyricsgenius.Genius with some options
genius = lyricsgenius.Genius(
    GENIUS_API_TOKEN,
    timeout=15,
    retries=3,
    sleep_time=0.25,  # small pause between each page scrape
    excluded_terms=["(Remix)", "(Live)"],  # Exclude these terms from song titles
    skip_non_songs=True,  # Skip non-song entries (e.g., interviews)
  # Remove section headers like "Verse", "Chorus"
)


# -----------------------------
# RATE LIMIT HANDLER
# -----------------------------
def with_rate_limit_handling(api_function):
    """Decorator to handle rate limit errors with exponential backoff"""
    def wrapper(*args, **kwargs):
        for attempt in range(MAX_RETRIES + 1):
            try:
                return api_function(*args, **kwargs)
            except Exception as e:
                error_str = str(e)
                # Check if it's a rate limit error
                if "429" in error_str and attempt < MAX_RETRIES:
                    # Calculate backoff time with jitter
                    backoff_time = INITIAL_BACKOFF * (2 ** attempt) + random.uniform(1, 5)
                    print(f"\nRate limit exceeded. Waiting {backoff_time:.1f} seconds before retry {attempt+1}/{MAX_RETRIES}")
                    time.sleep(backoff_time)
                else:
                    if "429" in error_str:
                        print(f"\nRate limit exceeded after {MAX_RETRIES} retries. Consider increasing wait time.")
                    raise
    return wrapper



# -----------------------------
# HELPER FUNCTION: fetch_artist_lyrics
# -----------------------------
@with_rate_limit_handling
def search_artist(artist_name, max_songs):
    """Search for an artist with rate limit handling"""
    return genius.search_artist(artist_name, max_songs=max_songs, sort="popularity",get_full_info=False)

@with_rate_limit_handling
def search_song(title, artist):
    """Search for a song with rate limit handling"""
    return genius.search_song(title=title, artist=artist, get_full_info=False)


# Add this function after your imports and before the GENIUS CLIENT section


def fetch_artist_lyrics(artist_name, max_songs=SONGS_PER_ARTIST):
    """
    Fetch up to max_songs tracks for `artist_name`, returning a list of dicts
    """
    songs_data = []
    try:
        # Search for the artist with rate limit handling
        artist_obj = search_artist(artist_name, max_songs)
        
        if artist_obj is None or not artist_obj.songs:
            print(f"  → No songs found for artist: {artist_name}")
            return songs_data

        for song in artist_obj.songs:
            title = song.title.strip()
            lyrics = song.lyrics.strip()
            
            # Skip extremely short lyrics (e.g., < 20 chars)
            if len(lyrics) < 20:
                continue
            songs_data.append({
                "artist": artist_name,
                "song_title": title,
                "lyrics": lyrics
            })
            
    except Exception as e:
        print(f"ERROR: Could not search for artist [{artist_name}]: {e}")
        
    return songs_data

def main():
    # 1) Read existing CSV (if any), so we don't re‐scrape duplicates
    if os.path.exists(OUTPUT_CSV):
        master_df = pd.read_csv(OUTPUT_CSV, encoding='utf-8')
        # master_df = safe_read_csv(OUTPUT_CSV)
        # Create a set of (artist, song_title) for quick "already scraped" checks
        existing_pairs = set(zip(master_df["artist"], master_df["song_title"]))
        
        # Check which artists have already met their quota
        artist_song_counts = master_df.groupby('artist').size()
        complete_artists = set(artist_song_counts[artist_song_counts >= SONGS_PER_ARTIST].index)
        incomplete_artists = set(artist_song_counts[artist_song_counts < SONGS_PER_ARTIST].index)
        
        print(f"Loaded {len(master_df)} existing rows from {OUTPUT_CSV}")
        print(f"Complete artists (>= {SONGS_PER_ARTIST} songs): {len(complete_artists)}")
        print(f"Incomplete artists (< {SONGS_PER_ARTIST} songs): {len(incomplete_artists)}")
    else:
        master_df = pd.DataFrame(columns=["artist", "song_title", "lyrics"])
        existing_pairs = set()
        complete_artists = set()
        incomplete_artists = set()
        print(f"No existing CSV found. A new one will be created: {OUTPUT_CSV}")

    # 2) Read artist list
    with open(ARTIST_LIST_PATH, "r", encoding="utf-8") as f:
        artists = [line.strip() for line in f if line.strip()]
    print(f"Read {len(artists)} artists from {ARTIST_LIST_PATH}")

    # 3) Filter artists: skip complete ones, include incomplete and new ones
    artists_to_scrape = [artist for artist in artists if artist not in complete_artists]
    skipped_count = len(artists) - len(artists_to_scrape)
    
    print(f"Will scrape {len(artists_to_scrape)} artists (skipping {skipped_count} completed artists)")
    if incomplete_artists:
        print(f"Resuming scraping for {len(incomplete_artists)} incomplete artists")

    # 4) Loop over each artist that needs scraping
    for idx, artist_name in enumerate(artists_to_scrape, 1):
        # Check if this is a resume case
        if artist_name in incomplete_artists:
            current_count = len([pair for pair in existing_pairs if pair[0] == artist_name])
            remaining_needed = SONGS_PER_ARTIST - current_count
            print(f"[{idx}/{len(artists_to_scrape)}] Resuming artist: {artist_name} (has {current_count}, needs {remaining_needed} more) ", end="")
        else:
            print(f"[{idx}/{len(artists_to_scrape)}] Scraping new artist: {artist_name} ", end="")
        
        fetched = fetch_artist_lyrics(artist_name, max_songs=SONGS_PER_ARTIST)

        # Filter out any (artist, song) pairs we already have
        new_rows = []
        for item in fetched:
            key = (item["artist"], item["song_title"])
            if key in existing_pairs:
                continue
            new_rows.append(item)
            existing_pairs.add(key)

        # 5) Append new_rows to master_df (and save immediately)
        if new_rows:
            new_df = pd.DataFrame(new_rows)
            master_df = pd.concat([master_df, new_df], ignore_index=True)

            # Sort by artist for better organization
            master_df = master_df.sort_values(['artist', 'song_title']).reset_index(drop=True)

            # Save after each artist to avoid data loss if script crashes
            master_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
            print(f"→ Retrieved {len(new_rows)} new songs (total now {len(master_df)})")
        else:
            print("→ No new songs found or all songs already exist.")

        # 6) Sleep to avoid hitting rate limits
        time.sleep(SLEEP_BETWEEN_ARTISTS)

    # Final sorting and statistics
    master_df = master_df.sort_values(['artist', 'song_title']).reset_index(drop=True)
    master_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')

    print("\nScraping complete.")
    print(f"Final row count: {len(master_df)}")
    print(f"Distinct artists in CSV: {master_df['artist'].nunique()}")
    print(f"Distinct songs in CSV: {master_df['song_title'].nunique()}")
    
    # Show final artist statistics
    final_artist_counts = master_df.groupby('artist').size().sort_values(ascending=False)
    print(f"\nTop 10 artists by song count:")
    print(final_artist_counts.head(10))
    
    # Show artists that still need more songs
    incomplete_final = final_artist_counts[final_artist_counts < SONGS_PER_ARTIST]
    if len(incomplete_final) > 0:
        print(f"\nArtists still needing more songs ({len(incomplete_final)} total):")
        print(incomplete_final.head(10))

if __name__ == "__main__":
    main()

Will fetch up to 10 songs per artist
Will sleep 0.25 seconds between artist requests
Using initial backoff of 5s with 5 max retries
No existing CSV found. A new one will be created: ../Data/scraped_lyrics.csv
Read 300 artists from artists.txt
Will scrape 300 artists (skipping 0 completed artists)
[1/300] Scraping new artist: Taylor Swift Searching for songs by Taylor Swift...



d:\Code\LabResearch\Topic-Modeling-Recommendation\.venv311\Lib\site-packages\lyricsgenius\genius.py:589: FutureWarning: The constructor signature will change in a future version. It will change to Song(lyrics, body) instead of Song(client, json_dict, lyrics).
  song = Song(self, song_info, lyrics)


Song 1: "All Too Well (10 Minute Version) (Taylor’s Version) [From The Vault]"
"All Too Well (10 Minute Version) (Taylor’s Version) [Live Acoustic]" is not valid. Skipping.
Song 2: "Fortnight"


KeyboardInterrupt: 

In [ ]:
import lyricsgenius
from dotenv import load_dotenv
import os

load_dotenv()
GENIUS_API_TOKEN = os.getenv("GENIUS_API_TOKEN")
genius = lyricsgenius.Genius(
    GENIUS_API_TOKEN,
    timeout=15,
    retries=3,
    sleep_time=0.25,  # small pause between each page scrape
    # Exclude these terms from song titles
    excluded_terms=["(Remix)", "(Live)"],
    skip_non_songs=True,  # Skip non-song entries (e.g., interviews)
    # Remove section headers like "Verse", "Chorus"
    remove_section_headers=False
)

In [ ]:
# Clean lyrics by removing metadata and save to new CSV
import pandas as pd
import re
import os

def clean_lyrics_metadata(lyrics_text):
    """
    Clean lyrics by finding the first structural marker [Something] and keeping everything from there.
    
    Args:
        lyrics_text (str): Raw lyrics text with metadata
    
    Returns:
        str: Cleaned lyrics starting from first structural marker
    """
    if pd.isna(lyrics_text) or not lyrics_text:
        return ""
    
    text = str(lyrics_text)
    
    # Find the first occurrence of specific structural markers containing: Intro, Chorus, or Verse 1
    # This pattern allows additional characters within brackets but requires one of the three key terms
    pattern = r'\[(Intro|Chorus|Verse|Pre-Chorus|Bridge).*?\]'
    match = re.search(pattern, text)
    
    if match:
        # Get the starting position of the first structural marker
        start_index = match.start()
        # Return everything from this point onward
        cleaned_text = text[start_index:].strip()
        return cleaned_text
    else:
        # If no structural marker found, return the original text
        # (this handles edge cases where songs might not have standard structure)
        return text.strip()

# Load the scraped lyrics
print("📁 Loading scraped lyrics data...")
input_csv = "scraped_lyrics.csv"

if not os.path.exists(input_csv):
    print(f"❌ Error: {input_csv} not found!")
    print("💡 Make sure you've run the scraping process first.")
else:
    df = pd.read_csv(input_csv, encoding='utf-8')
    print(f"✅ Loaded {len(df)} songs from {input_csv}")
    
    # Show some examples before cleaning
    print(f"\n📝 Sample lyrics BEFORE cleaning:")
    for i in range(min(3, len(df))):
        lyrics = df.iloc[i]['lyrics']
        print(f"\n🎵 {df.iloc[i]['artist']} - {df.iloc[i]['song_title']}")
        print(f"   Original (first 150 chars): {lyrics[:150]}...")
        
        # Find first structural marker containing Intro, Chorus, or Verse 1
        pattern = r'\[.*?(Intro|Chorus|Verse 1).*?\]'
        match = re.search(pattern, lyrics)
        if match:
            start_pos = match.start()
            before_marker = lyrics[:start_pos]
            print(f"   Metadata to remove: '{before_marker.strip()}'")
            print(f"   First marker found: '{match.group()}'")
        else:
            print(f"   No structural marker found - keeping full text")
    
    # Apply cleaning function
    print(f"\n🧹 Cleaning lyrics metadata...")
    df['lyrics_cleaned'] = df['lyrics'].apply(clean_lyrics_metadata)
    
    # Show results after cleaning
    print(f"\n📝 Sample lyrics AFTER cleaning:")
    for i in range(min(3, len(df))):
        original_length = len(str(df.iloc[i]['lyrics']))
        cleaned_length = len(str(df.iloc[i]['lyrics_cleaned']))
        chars_removed = original_length - cleaned_length
        
        print(f"\n🎵 {df.iloc[i]['artist']} - {df.iloc[i]['song_title']}")
        print(f"   Cleaned (first 150 chars): {df.iloc[i]['lyrics_cleaned'][:150]}...")
        print(f"   Length: {original_length} → {cleaned_length} chars (removed {chars_removed})")
    
    # Calculate cleaning statistics
    original_lengths = [len(str(lyrics)) for lyrics in df['lyrics']]
    cleaned_lengths = [len(str(lyrics)) for lyrics in df['lyrics_cleaned']]
    
    total_chars_removed = sum(original_lengths) - sum(cleaned_lengths)
    avg_chars_removed = total_chars_removed / len(df)
    
    print(f"\n📊 Cleaning Statistics:")
    print(f"   Total characters removed: {total_chars_removed:,}")
    print(f"   Average characters removed per song: {avg_chars_removed:.1f}")
    print(f"   Percentage of text removed: {(total_chars_removed / sum(original_lengths)) * 100:.1f}%")
    
    # Check for songs where no cleaning occurred (no structural markers)
    no_change_count = sum(1 for i in range(len(df)) if df.iloc[i]['lyrics'] == df.iloc[i]['lyrics_cleaned'])
    print(f"   Songs with no structural markers: {no_change_count} ({no_change_count/len(df)*100:.1f}%)")
    
    # Save to new CSV
    output_csv = OUTPUT_FOLDER + "scraped_lyrics_no_metadata.csv"
    
    # Create new DataFrame with cleaned lyrics
    cleaned_df = df[['artist', 'song_title', 'lyrics_cleaned']].copy()
    cleaned_df = cleaned_df.rename(columns={'lyrics_cleaned': 'lyrics'})
    
    # Save cleaned data
    cleaned_df.to_csv(output_csv, index=False, encoding='utf-8')
    
    print(f"\n💾 Saved cleaned lyrics to: {output_csv}")
    print(f"✅ Processing complete!")
    
    # Show final dataset info
    print(f"\n📋 Final Dataset Info:")
    print(f"   File: {output_csv}")
    print(f"   Songs: {len(cleaned_df):,}")
    print(f"   Artists: {cleaned_df['artist'].nunique():,}")
    print(f"   Columns: {list(cleaned_df.columns)}")
    
    # Verify the cleaning worked by showing first few characters of cleaned lyrics
    print(f"\n🔍 Verification - First characters of cleaned lyrics:")
    for i in range(min(5, len(cleaned_df))):
        lyrics_start = cleaned_df.iloc[i]['lyrics'][:50]
        print(f"   {i+1}. {lyrics_start}...")
        
    print(f"\n🎉 Metadata removal complete! Use '{output_csv}' for your topic modeling.")

📁 Loading scraped lyrics data...
✅ Loaded 3000 songs from scraped_lyrics.csv

📝 Sample lyrics BEFORE cleaning:

🎵 21 Savage - Bank Account
   Original (first 150 chars): 276 ContributorsTranslationsFrançaisРусский (Russian)PortuguêsBank Account Lyrics“Bank Account” sees 21 Savage rapping about his wealth amongst other ...
   Metadata to remove: '276 ContributorsTranslationsFrançaisРусский (Russian)PortuguêsBank Account Lyrics“Bank Account” sees 21 Savage rapping about his wealth amongst other things over a dark beat produced by Metro Boomin and 21 Savage himself.

The track samples Coleridge-Taylor Perkinson’s 1974 song… Read More'
   First marker found: '[Intro]'

🎵 21 Savage - Ghostface Killers
   Original (first 150 chars): 190 ContributorsTranslationsFrançaisPortuguêsРусский (Russian)Ghostface Killers Lyrics21 Savage, Offset, and Travis Scott open up Without Warning with...
   Metadata to remove: '190 ContributorsTranslationsFrançaisPortuguêsРусский (Russian)Ghostface Killers Lyric

# 🔍 Regex Pattern Breakdown: `r'\[.*?(Intro|Chorus|Verse 1).*?\]'`

This regex pattern is designed to find the **first structural marker** in song lyrics that contains one of three specific terms: `Intro`, `Chorus`, or `Verse 1`. It allows for additional characters within the brackets. Let's break it down piece by piece:

## 📋 **Complete Pattern Analysis**

```regex
r'\[.*?(Intro|Chorus|Verse 1).*?\]'
```

### 🧩 **Component Breakdown**

| Component | Description | What it Matches |
|-----------|-------------|-----------------|
| `r'...'` | Raw string | Prevents Python from interpreting backslashes as escape characters |
| `\[` | Escaped opening bracket | Literal `[` character |
| `.*?` | Non-greedy any character | Zero or more characters (non-greedy match) |
| `(Intro\|Chorus\|Verse 1)` | Alternation group | Exactly one of: "Intro" OR "Chorus" OR "Verse 1" |
| `.*?` | Non-greedy any character | Zero or more characters (non-greedy match) |
| `\]` | Escaped closing bracket | Literal `]` character |

### 🎯 **Step-by-Step Matching Process**

1. **`\[`** - Must start with an opening bracket `[`
2. **`.*?`** - Match any characters (non-greedy) - allows prefix text
3. **`(Intro|Chorus|Verse 1)`** - Must contain one of these three terms
4. **`.*?`** - Match any characters (non-greedy) - allows suffix text
5. **`\]`** - Must end with a closing bracket `]`

### ✅ **Examples of What This Matches**

```text
[Intro]                    ✅ - Basic marker
[Chorus]                   ✅ - Basic marker
[Verse 1]                  ✅ - Basic marker
[Intro:]                   ✅ - With colon
[Chorus - Repeated]        ✅ - With additional info
[Verse 1: Drake]           ✅ - With artist info
[Pre-Intro]               ✅ - With prefix
[Chorus (4x)]             ✅ - With repetition info
[Main Chorus]             ✅ - With prefix
[Verse 1 - Jay-Z]         ✅ - With artist and dash
```

### ❌ **Examples of What This DOESN'T Match**

```text
[Verse 2]                 ❌ - Different verse number
[Pre-Chorus]              ❌ - Doesn't contain any of the three terms
[Bridge]                  ❌ - Not one of the three specified terms
[Outro]                   ❌ - Not one of the three specified terms
[Hook]                    ❌ - Not one of the three specified terms
[intro]                   ❌ - Case sensitive (lowercase)
[CHORUS]                  ❌ - Case sensitive (uppercase)
[123]                     ❌ - No matching terms
```

### 🎵 **Why This Flexible Pattern Works**

1. **Flexible Structure**: Handles various formatting styles with prefixes/suffixes
2. **Key Term Focus**: Still requires one of the three core structural terms
3. **Real-world Compatible**: Matches actual lyric formatting variations
4. **Non-greedy Matching**: `.*?` ensures we get the shortest possible match

### 🔧 **Pattern Evolution**

```python
# Most restrictive - exact matches only
r'\[(Intro|Chorus|Verse 1})\]'

# Current flexible pattern - allows additional characters
r'\[.*?(Intro|Chorus|Verse 1).*?\]'

# Most permissive - any brackets with letters
r'\[[^\]]*[a-zA-Z][^\]]*\]'
```

### ⚠️ **Important Notes**

- **Case Sensitive**: Still requires exact capitalization (`Intro`, not `intro`)
- **Key Terms Required**: Must contain one of the three specified terms
- **Non-greedy**: Uses `.*?` to avoid matching too much text
- **Flexible Format**: Allows real-world variations in bracket formatting

This balanced approach provides **flexibility** while maintaining **precision** for the three most important structural markers!